# RCDNet — Change Detection Visualizer
Each row shows: **Before (2021)** | **After (2024)** | **Change mask overlay**

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image

ROOT = Path(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
DATA_DIR     = ROOT / 'showcase' / 'data'
RESULTS_DIR  = ROOT / 'showcase' / 'results' / 'change_maps'

# Denormalize for display (SECOND training stats)
MEAN = np.array([0.439, 0.447, 0.459])
STD  = np.array([0.193, 0.183, 0.189])

CLASSES = sorted([d.name for d in RESULTS_DIR.iterdir() if d.is_dir()])
print('Available classes:')
for i, c in enumerate(CLASSES):
    n = len(list((RESULTS_DIR / c).glob('*.png')))
    print(f'  [{i}] {c}  ({n} detections)')

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
CLASS_NAME   = 'building'   # change to any class name above
N_ROWS       = 8            # number of patches to display
SEED         = 42           # set None for a new random selection each run
OVERLAY_ALPHA = 0.45        # transparency of the red change overlay
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
def load_rgb(path):
    """Load image as float32 [0,1] array."""
    return np.array(Image.open(path).convert('RGB'), dtype=np.float32) / 255.0

def overlay_mask(img_rgb, mask, color=(1.0, 0.15, 0.15), alpha=0.45):
    """Blend a binary mask (H,W) as a colored overlay onto img_rgb (H,W,3)."""
    out = img_rgb.copy()
    m = mask > 0
    for c, col in enumerate(color):
        out[m, c] = out[m, c] * (1 - alpha) + col * alpha
    return out

# Collect patches that have a detection for the chosen class
cls_dir = RESULTS_DIR / CLASS_NAME
patch_ids = sorted([p.stem for p in cls_dir.glob('*.png')])

if SEED is not None:
    random.seed(SEED)
sample = random.sample(patch_ids, min(N_ROWS, len(patch_ids)))
print(f'Showing {len(sample)} / {len(patch_ids)} patches for class "{CLASS_NAME}"')

In [ ]:
fig, axes = plt.subplots(len(sample), 3,
                          figsize=(15, 5 * len(sample)),
                          squeeze=False)

col_titles = ['Before (2021)', 'After (2024)', f'Changes: {CLASS_NAME.replace("_", " ").title()}']
for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontsize=14, fontweight='bold', pad=10)

for row, pid in enumerate(sample):
    img_a = load_rgb(DATA_DIR / 'A' / f'{pid}.png')
    img_b = load_rgb(DATA_DIR / 'B' / f'{pid}.png')
    mask  = np.array(Image.open(cls_dir / f'{pid}.png'))

    changed_px = int((mask > 0).sum())
    changed_pct = changed_px / (mask.shape[0] * mask.shape[1]) * 100

    overlay = overlay_mask(img_b, mask, alpha=OVERLAY_ALPHA)

    axes[row][0].imshow(img_a)
    axes[row][1].imshow(img_b)
    axes[row][2].imshow(overlay)

    # Red patch legend on the overlay panel
    patch = mpatches.Patch(color=(1.0, 0.15, 0.15), label=f'{changed_px:,} px ({changed_pct:.1f}%)')
    axes[row][2].legend(handles=[patch], loc='lower right', fontsize=9,
                         framealpha=0.7, edgecolor='white')

    # Row label
    axes[row][0].set_ylabel(pid, fontsize=11, rotation=0, labelpad=55,
                             va='center', ha='right', fontweight='bold')

    for ax in axes[row]:
        ax.axis('off')

plt.suptitle(f'Change Detection — {CLASS_NAME.replace("_", " ").title()}\n'
             f'Saint-Denis / Village Olympique · 2021 → 2024',
             fontsize=16, fontweight='bold', y=1.002)

plt.tight_layout()
out_path = ROOT / 'showcase' / 'visualizations' / f'{CLASS_NAME}_grid.png'
plt.savefig(out_path, dpi=120, bbox_inches='tight')
print(f'Saved → {out_path}')
plt.show()

In [ ]:
# ── All-classes summary grid (1 sample per class) ─────────────────────────────
# Shows the patch with the most changed pixels for each class.

def top_patch(cls_name):
    """Return (patch_id, pixel_count) for the patch with most detected change."""
    best_id, best_n = None, 0
    for p in (RESULTS_DIR / cls_name).glob('*.png'):
        n = int((np.array(Image.open(p)) > 0).sum())
        if n > best_n:
            best_id, best_n = p.stem, n
    return best_id, best_n

fig2, axes2 = plt.subplots(len(CLASSES), 3,
                            figsize=(15, 5 * len(CLASSES)),
                            squeeze=False)

for ax, title in zip(axes2[0], col_titles):
    ax.set_title(title, fontsize=14, fontweight='bold', pad=10)

for row, cls_name in enumerate(CLASSES):
    pid, n_px = top_patch(cls_name)
    if pid is None:
        for ax in axes2[row]: ax.axis('off')
        continue

    img_a = load_rgb(DATA_DIR / 'A' / f'{pid}.png')
    img_b = load_rgb(DATA_DIR / 'B' / f'{pid}.png')
    mask  = np.array(Image.open(RESULTS_DIR / cls_name / f'{pid}.png'))
    overlay = overlay_mask(img_b, mask, alpha=OVERLAY_ALPHA)

    axes2[row][0].imshow(img_a)
    axes2[row][1].imshow(img_b)
    axes2[row][2].imshow(overlay)

    pct = n_px / (mask.shape[0] * mask.shape[1]) * 100
    patch = mpatches.Patch(color=(1.0, 0.15, 0.15), label=f'{n_px:,} px ({pct:.1f}%)')
    axes2[row][2].legend(handles=[patch], loc='lower right', fontsize=9,
                          framealpha=0.7, edgecolor='white')

    label = cls_name.replace('_', ' ').title()
    axes2[row][0].set_ylabel(f'{label}\n{pid}', fontsize=10, rotation=0,
                              labelpad=80, va='center', ha='right', fontweight='bold')

    for ax in axes2[row]:
        ax.axis('off')

plt.suptitle('Best detection per class — Saint-Denis / Village Olympique · 2021 → 2024',
             fontsize=16, fontweight='bold', y=1.002)

plt.tight_layout()
out2 = ROOT / 'showcase' / 'visualizations' / 'all_classes_best.png'
plt.savefig(out2, dpi=120, bbox_inches='tight')
print(f'Saved → {out2}')
plt.show()